# 4. Coherence in a double dot and choosing an approximation

**Learning goals.** In this tutorial you will:

- build a serial double dot with coherent interdot tunnelling;
- read populations *and* coherences out of a QmeQ solution;
- compare the Pauli, Lindblad, Redfield, and 1vN approaches on the same model;
- see the Pauli equation fail by more than an order of magnitude, and understand the criterion that predicts the failure; and
- validate a coherent calculation against an analytical large-bias formula.

Tutorials 1&ndash;3 used the Pauli master equation, which propagates only the probabilities of dot eigenstates. This tutorial shows when that is not enough. We keep $\hbar=k_\mathrm{B}=|e|=1$.

## The physical question

Take two single-particle states, one in each half of a double dot. The left dot couples only to the left reservoir, the right dot only to the right reservoir, and the two dots are connected by a coherent interdot amplitude $\Omega$:

$$H_\mathrm{dot}=\varepsilon_1 d_1^\dagger d_1+\varepsilon_2 d_2^\dagger d_2+\Omega\,(d_1^\dagger d_2+d_2^\dagger d_1),$$

with a strong charging energy $U$ that keeps at most one electron on the double dot. At zero detuning $\varepsilon_1=\varepsilon_2=\varepsilon$, the one-electron eigenstates are the bonding and antibonding combinations

$$|\pm\rangle=\tfrac{1}{\sqrt2}\left(|1\rangle\pm|2\rangle\right),\qquad E_\pm=\varepsilon\pm\Omega,$$

split by $2\Omega$. Both eigenstates are delocalized, so *both* couple to *both* reservoirs. That is the structural difference from the models in Tutorials 1&ndash;3, where each single-particle state had one reservoir on each side and spin was conserved.

When two eigenstates couple to the same reservoir and their splitting is not large compared with the tunnelling rate,

$$2\Omega\lesssim\Gamma,$$

the reservoirs drive a coherent superposition of $|+\rangle$ and $|-\rangle$. The stationary state then has off-diagonal density-matrix elements $\rho_{+-}$ of the same order as the populations, and any equation that keeps only populations is uncontrolled.

**Prediction before calculating.** For $2\Omega\gg\Gamma$ all first-order approaches should agree. For $2\Omega\ll\Gamma$ the Pauli result should separate from the three approaches that keep coherences, and it should be *too large*: dropping $\rho_{+-}$ removes destructive interference between the two transport paths.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import qmeq

print(qmeq.get_backend_status())

In [ ]:
# Double-dot and reservoir parameters
gamma = 0.5
tunnel_amplitude = np.sqrt(gamma / (2 * np.pi))
temperature = 1.0
bias = 2.0
charging_energy = 1.0e3  # keeps the double dot in the 0/1 electron subspace
bandwidth = 5.0e3

def make_double_dot(kerntype, omega, detuning=0.0,
                    voltage=bias, temp=temperature):
    return qmeq.Builder(
        nsingle=2,
        # Only the (0, 1) element is given: QmeQ adds its Hermitian conjugate.
        hsingle={(0, 0): detuning / 2, (1, 1): -detuning / 2, (0, 1): omega},
        coulomb={(0, 1, 1, 0): charging_energy},
        nleads=2,
        # Lead 0 couples to dot 1 only, lead 1 to dot 2 only.
        tleads={(0, 0): tunnel_amplitude, (1, 1): tunnel_amplitude},
        mulst={0: voltage / 2, 1: -voltage / 2},
        tlst={0: temp, 1: temp},
        dband=bandwidth,
        kerntype=kerntype,
    )

**A note on Hermiticity.** QmeQ builds a Hermitian dot Hamiltonian from the elements you supply, so giving `(0, 1): omega` is enough — the matching $\Omega^*d_2^\dagger d_1$ term is added automatically. Supplying `{(0, 1): omega, (1, 0): np.conj(omega)}` double-counts the interdot coupling. The same rule applies to `tleads` and to the Coulomb elements.

QmeQ diagonalizes the dot Hamiltonian on the first `solve` call and works in the resulting eigenbasis — here the $|\pm\rangle$ basis. As in Tutorial 2, `solve(masterq=False)` performs that diagonalization on its own:

In [ ]:
system = make_double_dot("1vN", omega=0.2)
system.solve(masterq=False)  # diagonalize the dot without solving transport

for charge, states in enumerate(system.si.statesdm):
    for state in states:
        print(f"charge {charge}, state {state}: energy = {system.Ea[state]: .4f}")

## Reading populations and coherences

The vector `system.phi0` holds the stationary reduced density matrix. Only elements between states of equal charge survive, and QmeQ stores them in a compact real form:

- the first `system.si.npauli` entries are the populations $\rho_{bb}$;
- the remaining entries are the real parts of the coherences $\rho_{bb'}$, followed by their imaginary parts, offset by `si.ndm0 - si.npauli`.

The dictionary `system.si.inddm0` maps each stored index to its state pair $(b,b')$, so the block of the density matrix belonging to one charge state can be rebuilt directly. Approaches that keep only populations — the Pauli equation here — return a shorter `phi0`, which the helper below detects.

In [ ]:
def density_matrix(system, charge):
    """Stationary density matrix of the given charge sector, as a matrix."""
    si = system.si
    states = si.statesdm[charge]
    position = {state: index for index, state in enumerate(states)}
    keeps_coherences = len(system.phi0) > si.npauli
    imaginary_offset = si.ndm0 - si.npauli

    rho = np.zeros((len(states), len(states)), dtype=complex)
    for index, (b, bp) in si.inddm0.items():
        if b not in position or bp not in position:
            continue
        if b == bp:
            rho[position[b], position[b]] = system.phi0[index]
        elif keeps_coherences:
            element = system.phi0[index] + 1j * system.phi0[index + imaginary_offset]
            rho[position[b], position[bp]] = element
            rho[position[bp], position[b]] = np.conj(element)
    return rho

for kerntype in ["Pauli", "1vN"]:
    for omega in [0.2, 2.0]:
        system = make_double_dot(kerntype, omega)
        system.solve()
        rho = density_matrix(system, charge=1)
        print(
            f"{kerntype:5s} 2*Omega/Gamma = {2 * omega / gamma:4.1f}: "
            f"populations = {np.round(rho.diagonal().real, 4)}, "
            f"|coherence| = {abs(rho[0, 1]):.4f}"
        )

For $2\Omega=0.8\Gamma$ the coherence reaches about half of $\sqrt{\rho_{++}\rho_{--}}$, its largest possible value; for $2\Omega=8\Gamma$ it has all but vanished. The Pauli solution reports no coherence at all, by construction.

## Comparing four first-order approaches

All four first-order approaches solve for the same stationary state of the same model; they differ in which terms of the kernel they retain.

- **Pauli** keeps populations and golden-rule rates between eigenstates.
- **Lindblad** keeps coherences in a form that guarantees a positive density matrix.
- **Redfield** and **1vN** keep coherences with different treatments of the energy dependence of the reservoir correlation functions.

`system.kerntype` can be reassigned on an existing system, which rebuilds the solver but keeps the model. We sweep $\Omega$ from deep in the coherent regime to the incoherent one.

In [ ]:
omegas = np.geomspace(0.02, 4.0, 40)
methods = ["Pauli", "Lindblad", "Redfield", "1vN"]
currents = {method: np.empty_like(omegas) for method in methods}

for method in methods:
    system = make_double_dot(method, omegas[0])
    for index, omega in enumerate(omegas):
        system.change(hsingle={(0, 0): 0.0, (1, 1): 0.0, (0, 1): omega})
        system.solve()
        currents[method][index] = system.current[0]
        assert np.isclose(np.sum(system.current), 0.0, atol=1e-10)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for method, style in zip(methods, ["k-", "C0--", "C1-.", "C2:"]):
    axes[0].semilogx(2 * omegas / gamma, currents[method] / gamma, style, label=method)
    axes[1].loglog(
        2 * omegas / gamma, currents[method] / currents["1vN"], style, label=method
    )
axes[1].axhline(1.0, color="0.7", lw=0.8)
axes[0].set(xlabel="$2\\Omega/\\Gamma$", ylabel="$I_L/\\Gamma$",
            title="Current through the double dot")
axes[1].set(xlabel="$2\\Omega/\\Gamma$", ylabel="$I_L/I_L^{1\\mathrm{vN}}$",
            title="Ratio to the 1vN result")
for axis in axes:
    axis.legend()
fig.tight_layout()

for index in [0, len(omegas) // 2, len(omegas) - 1]:
    print(f"2*Omega/Gamma = {2 * omegas[index] / gamma:6.2f}: "
          f"Pauli / 1vN = {currents['Pauli'][index] / currents['1vN'][index]:7.2f}, "
          f"Lindblad / 1vN = {currents['Lindblad'][index] / currents['1vN'][index]:.3f}, "
          f"Redfield / 1vN = {currents['Redfield'][index] / currents['1vN'][index]:.3f}")

The prediction holds. For $2\Omega\gtrsim\Gamma$ all four curves stay within about ten to fifteen percent of each other. Below that the Pauli current separates and diverges as $\Omega\rightarrow0$: at $2\Omega=0.08\Gamma$ it is forty times the coherent result. That is not a small correction and not a numerical artifact — it is the interference between the two transport paths, which the Pauli equation cannot represent. The Pauli current stays finite because it treats the two eigenstates as independently populated channels, while the coherent solutions correctly send $I\rightarrow0$ as the dots decouple.

Among the coherent approaches, Redfield and 1vN agree to about one percent, while Lindblad runs six to thirteen percent below them because its rates are evaluated in the form that guarantees a positive density matrix. Disagreement of that size between valid approximations is expected; it measures the uncertainty of the first-order description rather than indicating a defect.

## An analytical check in the large-bias limit

Coherent transport through a serial double dot has a closed-form solution when the bias is much larger than all dot energies, the band is wide, and only one electron is allowed on the dot. For symmetric couplings $\Gamma_L=\Gamma_R=\Gamma$ and detuning $\varepsilon_{12}=\varepsilon_1-\varepsilon_2$,

$$I=\frac{\Omega^2\Gamma}{3\Omega^2+\Gamma^2/4+\varepsilon_{12}^2}.$$

This is the standard rate-equation result for a coherently coupled double dot at infinite bias. It is an independent check of the solver: the numerics must reproduce it in the regime where it applies, including the way the current *saturates* at large $\Omega$ instead of growing.

The formula is derived in the wide-band limit and without the principal-value (Lamb-shift) terms that renormalize the dot energies. QmeQ selects that treatment with the integral type `itype`: the default `itype=0` keeps the Lamb shift through a digamma function, while `itype=2` neglects it — exactly the assumptions behind the formula. We therefore compare both.

In [ ]:
large_bias, cold = 100.0, 0.05

def double_dot_large_bias(kerntype, omega, detuning, itype):
    system = make_double_dot(kerntype, omega, detuning,
                             voltage=large_bias, temp=cold)
    system.itype = itype
    return system

def analytical_current(omega, detuning=0.0):
    return omega**2 * gamma / (3 * omega**2 + gamma**2 / 4 + detuning**2)

header = ("2*Omega/Gamma", "detuning", "Lindblad", "1vN", "Lindblad", "formula")
print(f"{header[0]:>13} {header[1]:>9} {header[2]:>11} {header[3]:>11}"
      f" {header[4]:>11} {header[5]:>11}")
print(f"{'':>13} {'':>9} {'itype=2':>11} {'itype=2':>11} {'itype=0':>11} {'':>11}")
for omega, detuning in [(0.1, 0.0), (0.5, 0.0), (2.0, 0.0), (0.5, 1.0), (0.5, 2.0)]:
    exact = analytical_current(omega, detuning)
    results = {}
    for label, kerntype, itype in [("lindblad", "Lindblad", 2),
                                   ("first_vn", "1vN", 2),
                                   ("lamb", "Lindblad", 0)]:
        system = double_dot_large_bias(kerntype, omega, detuning, itype)
        system.solve()
        results[label] = system.current[0]
    print(f"{2 * omega / gamma:13.1f} {detuning:9.1f} {results['lindblad']:11.7f}"
          f" {results['first_vn']:11.7f} {results['lamb']:11.7f} {exact:11.7f}")
    # Under the assumptions of the formula both coherent kernels are exact.
    assert np.isclose(results["lindblad"], exact, rtol=1e-9)
    assert np.isclose(results["first_vn"], exact, rtol=1e-9)
    # Keeping the Lamb shift shifts the current by a small, finite amount.
    assert np.isclose(results["lamb"], exact, rtol=1e-2)

print("large-bias limit reproduced by the coherent approaches")

Both coherent kernels reproduce the formula to machine precision once the Lamb shift is switched off, so the coherent part of the calculation is verified independently of QmeQ. Keeping the Lamb shift (`itype=0`, the default) changes the current by up to about one percent at finite detuning: a real physical effect of the reservoirs on the dot energies, not an error.

## When the choice does not matter

Coherences only matter if the model can develop them. In the spinful single-orbital Anderson model of Tutorials 2 and 3, each spin couples to its own reservoir channels and spin is conserved, so no coherence between dot eigenstates is generated — even with a Zeeman field. All four approaches must then give the same current.

In [ ]:
U = 20.0
temp = 1.0
gam = 0.5
amplitude = np.sqrt(gam / (2 * np.pi))
zeeman = 7.5

def make_spinful(kerntype, gate, field=zeeman, bias=0.5):
    return qmeq.Builder(
        nsingle=2,
        hsingle={(0, 0): gate + field / 2, (1, 1): gate - field / 2},
        coulomb={(0, 1, 1, 0): U},
        nleads=4,
        tleads={(0, 0): amplitude, (1, 0): amplitude,
                (2, 1): amplitude, (3, 1): amplitude},
        mulst={0: bias / 2, 1: -bias / 2, 2: bias / 2, 3: -bias / 2},
        tlst={0: temp, 1: temp, 2: temp, 3: temp},
        dband=60.0,
        kerntype=kerntype,
    )

spinful_currents = {}
for method in methods:
    spinful = make_spinful(method, gate=-U / 2)
    spinful.solve()
    spinful_currents[method] = spinful.current[0] + spinful.current[2]

for method, value in spinful_currents.items():
    print(f"{method:9s} I_L = {value:.10e}")

reference = spinful_currents["1vN"]
for method, value in spinful_currents.items():
    assert np.isclose(value, reference, rtol=1e-8)

print("all four approaches agree when spin is a good quantum number")

This is worth remembering: agreement between approaches does not certify that the physics is captured, and disagreement does not identify which one is right. Here the four kernels agree because the model has a selection rule, not because first-order theory is exact.

## Excited states in a stability diagram

A Zeeman field splits the two single-particle states, so the doubly occupied state can be reached through two different one-electron levels. In a stability diagram this appears as extra lines running parallel to the Coulomb-diamond edges, marking the bias at which the *excited* transition enters the transport window.

Tutorial 3 obtained the differential conductance by differentiating a current map with `numpy.gradient`. Here we use the alternative: `system.add`, which increments the parameters that `system.change` sets, to evaluate a symmetric finite difference in bias at each point. It costs two solves per point but does not couple neighbouring grid points.

In [ ]:
# Modest grids keep the notebook quick; increase them for publication plots.
gate_values = np.linspace(-2.0 * U, 1.0 * U, 41)
bias_values = np.linspace(-1.5 * U, 1.5 * U, 41)
dV = 1.0e-3

def zeeman_conductance(field):
    system = make_spinful("1vN", gate=gate_values[0], field=field)
    conductance = np.empty((len(bias_values), len(gate_values)))
    for gate_index, gate in enumerate(gate_values):
        system.change(hsingle={(0, 0): gate + field / 2, (1, 1): gate - field / 2})
        system.solve(masterq=False)
        for bias_index, bias in enumerate(bias_values):
            system.change(mulst={0: bias / 2, 1: -bias / 2,
                                 2: bias / 2, 3: -bias / 2})
            system.solve(qdq=False)
            current_below = system.current[0] + system.current[2]
            system.add(mulst={0: dV / 2, 1: -dV / 2, 2: dV / 2, 3: -dV / 2})
            system.solve(qdq=False)
            current_above = system.current[0] + system.current[2]
            conductance[bias_index, gate_index] = (current_above - current_below) / dV
    return conductance

conductance_zero_field = zeeman_conductance(0.0)
conductance_split = zeeman_conductance(zeeman)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharex=True, sharey=True)
extent = [gate_values[0] / U, gate_values[-1] / U,
          bias_values[0] / U, bias_values[-1] / U]
for axis, data, title in zip(
    axes,
    [conductance_zero_field, conductance_split],
    ["$B=0$", f"$B={zeeman / U:.2f}\\,U$"],
):
    image = axis.imshow(data, origin="lower", aspect="auto", extent=extent,
                        cmap="viridis")
    fig.colorbar(image, ax=axis, label="$\\partial I_L/\\partial V$")
    axis.set(xlabel="$\\varepsilon/U$", title=title)
axes[0].set_ylabel("$V/U$")
fig.tight_layout()

At $B=0$ the conductance follows the diamond edges only. At finite $B$ each edge acquires a companion line offset by the Zeeman energy: transport through the excited spin state becomes available once $|V|$ exceeds the splitting. Reading such offsets off a measured diagram is the standard way to extract excited-state energies.

## Choosing an approximation

| approach | keeps coherences | order in $\Gamma$ | use it when |
| --- | --- | --- | --- |
| Pauli | no | first | eigenstates are well separated ($\Delta E\gg\Gamma$) or protected by a selection rule |
| Lindblad | yes | first | coherences matter and a positive density matrix is required |
| Redfield, 1vN | yes | first | coherences matter; 1vN retains more of the energy dependence of the reservoirs |
| RTD, 2vN | see Tutorial 6 | second | first-order transport is blocked, or $\Gamma$ is not small |

All rows share the first-order limitation $\Gamma\ll T$: they do not describe cotunnelling, level broadening, or Kondo correlations. A calculation that changes qualitatively between two of these approaches is telling you that the model sits at the edge of their common validity — the response is to check the criterion ($\Delta E$ versus $\Gamma$, $\Gamma$ versus $T$), not to pick the prettier curve.

## Exercises

1. Set `bias = 0.2` (below the temperature) and repeat the $\Omega$ sweep. Does the Pauli equation still fail in the same way, and does the failure still set in at $2\Omega\approx\Gamma$?
2. Add a detuning $\varepsilon_{12}=4\Gamma$ to the sweep. Predict first: detuning splits the eigenstates, so the coherent and incoherent descriptions should converge.
3. Raise `gamma` to `5.0` at fixed temperature. All four curves become unreliable, because $\Gamma\ll T$ is violated; Tutorial 6 shows how to test that with a second-order approach.
4. Recompute the split stability diagram with `Pauli` instead of `1vN`. The Zeeman lines survive, because this model has the selection rule established above.